# Feature Engineering

This notebook focuses on transforming the cleaned electricity demand dataset into a machine-learning-ready dataset. New features are engineered from existing variables to capture temporal patterns, renewable energy contributions, and historical demand behavior that may improve forecasting performance.

In [1]:
import pandas as pd

features_df = pd.read_csv(
    "../data/processed/master_clean.csv",
    parse_dates=["utc_timestamp", "time"]
)

features_df.shape

(50400, 21)

### Verifing Dataset Structure

In [2]:
features_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50400 entries, 0 to 50399
Data columns (total 21 columns):
 #   Column                              Non-Null Count  Dtype              
---  ------                              --------------  -----              
 0   utc_timestamp                       50400 non-null  datetime64[us, UTC]
 1   DE_load_actual_entsoe_transparency  50400 non-null  float64            
 2   DE_solar_generation_actual          50400 non-null  float64            
 3   DE_wind_generation_actual           50400 non-null  float64            
 4   DE_solar_capacity                   50400 non-null  float64            
 5   DE_wind_capacity                    50400 non-null  float64            
 6   DE_LU_price_day_ahead               17540 non-null  float64            
 7   time                                50400 non-null  datetime64[us, UTC]
 8   temperature_C                       50400 non-null  float64            
 9   humidity_pct                        50400 non-null

## Renewable Energy Features

Renewable energy features were created to represent the combined contribution of solar and wind generation and their share relative to electricity demand. These variables may help capture the influence of renewable energy availability on electricity demand patterns and market dynamics.

In [3]:
features_df['renewable_generation'] = (
    features_df['DE_solar_generation_actual']
    + features_df['DE_wind_generation_actual']
)

features_df['renewable_share_pct'] = (
    features_df['renewable_generation']
    / features_df['DE_load_actual_entsoe_transparency']
) * 100

features_df[
    [
        'renewable_generation',
        'renewable_share_pct'
    ]
].head()

,renewable_generation,renewable_share_pct
0,8923.0,21.683556
1,9125.0,22.735767
2,9141.0,23.374930
3,9234.0,23.820457
4,9302.0,23.887419


### Renewable Energy Features

Two renewable energy features were engineered to represent the contribution of renewable resources within the electricity system. Renewable generation was calculated as the sum of solar and wind generation, while renewable share was calculated as the percentage contribution of renewable generation relative to total electricity demand.

These features provide a compact representation of renewable energy availability and may serve as useful predictors in future forecasting models. They also enable analysis of renewable penetration and its interaction with electricity demand and market variables.

## Lagged Demand Features

Lagged demand features were created to capture temporal persistence and recurring consumption patterns. Electricity demand is often strongly influenced by recent demand levels as well as repeating daily and weekly behavioral cycles.

In [4]:
features_df['load_lag_1'] = (
    features_df['DE_load_actual_entsoe_transparency']
    .shift(1)
)

features_df['load_lag_24'] = (
    features_df['DE_load_actual_entsoe_transparency']
    .shift(24)
)

features_df['load_lag_168'] = (
    features_df['DE_load_actual_entsoe_transparency']
    .shift(168)
)

features_df[
    [
        'DE_load_actual_entsoe_transparency',
        'load_lag_1',
        'load_lag_24',
        'load_lag_168'
    ]
].head(10)

,DE_load_actual_entsoe_transparency,load_lag_1,load_lag_24,load_lag_168
0,41151.0,NaN,NaN,NaN
1,40135.0,41151.0,NaN,NaN
2,39106.0,40135.0,NaN,NaN
3,38765.0,39106.0,NaN,NaN
4,38941.0,38765.0,NaN,NaN
5,39045.0,38941.0,NaN,NaN
6,40206.0,39045.0,NaN,NaN
7,41133.0,40206.0,NaN,NaN
8,42963.0,41133.0,NaN,NaN
9,45088.0,42963.0,NaN,NaN


### Lagged Demand Features

Three lagged demand features were engineered to capture short-term and recurring temporal patterns in electricity consumption. The features represent demand in the previous hour (load_lag_1), the same hour on the previous day (load_lag_24), and the same hour during the previous week (load_lag_168).

These features were selected based on the exploratory analysis, where they exhibited the strongest relationships with current demand. The lagged variables enable forecasting models to leverage temporal persistence and recurring behavioral patterns that characterize electricity consumption.

## Missing Values Introduced by Feature Engineering

Lagged features require historical observations and therefore introduce missing values at the beginning of the dataset. The extent of these missing values must be evaluated before preparing the final modeling dataset.

In [5]:
features_df[
    [
        'load_lag_1',
        'load_lag_24',
        'load_lag_168'
    ]
].isnull().sum()

load_lag_1        1
load_lag_24      24
load_lag_168    168
dtype: int64

### Missing Values from Lag Features

The creation of lagged demand features introduced missing values at the beginning of the dataset because historical observations are unavailable for the earliest timestamps. Specifically, the one-hour, 24-hour, and 168-hour lag features introduced 1, 24, and 168 missing values, respectively.

Because the largest lag feature requires 168 hours of historical data, the first 168 observations cannot be used for model training when all lag features are included. However, this represents less than 1% of the dataset and is therefore unlikely to have a significant impact on model development.

## Preparing the Modeling Dataset

Rows containing missing values introduced by lagged features were removed to create a complete feature set suitable for machine learning. This dataset will serve as the primary input for electricity demand forecasting models.

In [6]:
model_df = features_df.dropna(
    subset=[
        'load_lag_1',
        'load_lag_24',
        'load_lag_168'
    ]
).copy()

print("Original shape:", features_df.shape)
print("Model shape:", model_df.shape)

Original shape: (50400, 26)
Model shape: (50232, 26)


### Creation of the Modeling Dataset

A modeling dataset was created by removing observations with missing values in the lagged demand features. Because the largest lag feature required 168 hours of historical data, the first 168 observations were excluded.

The resulting dataset contained 50,232 observations, representing more than 99% of the original data. This minimal reduction ensures that sufficient historical information is available for all lag-based predictors while preserving nearly the entire dataset for model development.

## Assessing Missing Values in Electricity Price Data

Electricity price is a potentially useful forecasting feature; however, missing observations may reduce the number of samples available for model training. The extent of missing values is evaluated before deciding whether to retain, impute, or exclude this variable.

In [7]:
model_df['DE_LU_price_day_ahead'].isnull().sum()

np.int64(32692)

### Missing Electricity Price Data

The electricity price feature contained 32,692 missing values out of 50,232 observations, corresponding to approximately 65% of the modeling dataset.

Because removing all rows with missing price data would substantially reduce the available sample size, the electricity price feature was not selected for the initial forecasting models. This decision preserves the majority of observations while avoiding the need for large-scale imputation.

Future work may investigate specialized approaches for incorporating electricity price data once a suitable strategy for handling missing values has been established.

In [8]:
model_df.columns.tolist()

['utc_timestamp',
 'DE_load_actual_entsoe_transparency',
 'DE_solar_generation_actual',
 'DE_wind_generation_actual',
 'DE_solar_capacity',
 'DE_wind_capacity',
 'DE_LU_price_day_ahead',
 'time',
 'temperature_C',
 'humidity_pct',
 'cloud_cover_pct',
 'wind_speed_ms',
 'precipitation_mm',
 'hour',
 'day_of_week',
 'month',
 'year',
 'is_weekend',
 'quarter',
 'week_of_year',
 'season',
 'renewable_generation',
 'renewable_share_pct',
 'load_lag_1',
 'load_lag_24',
 'load_lag_168']

### Verification of Engineered Features

The engineered dataset was verified to ensure that all newly created features were successfully added. The final feature set includes renewable energy metrics and lagged demand variables alongside the original weather, calendar, generation, and demand variables.

These engineered features provide additional information about historical demand behavior and renewable energy availability, both of which are expected to improve forecasting performance.

## Additional Feature Engineering

Following the initial feature engineering process, additional temporal and trend-based features were explored to further enhance the forecasting dataset.

### Rolling Demand Features

Rolling statistics were created to capture recent demand trends. A 24-hour rolling mean was calculated to represent the average electricity demand over the previous day, providing contextual information about short-term demand behavior.

In [9]:
features_df['rolling_mean_24'] = (
    features_df['DE_load_actual_entsoe_transparency']
    .rolling(window=24)
    .mean()
)

features_df[
    [
        'DE_load_actual_entsoe_transparency',
        'rolling_mean_24'
    ]
].head(30)

,DE_load_actual_entsoe_transparency,rolling_mean_24
0,41151.0,NaN
1,40135.0,NaN
2,39106.0,NaN
3,38765.0,NaN
4,38941.0,NaN
5,39045.0,NaN
6,40206.0,NaN
7,41133.0,NaN
8,42963.0,NaN
9,45088.0,NaN


### 24-Hour Rolling Mean Demand

A 24-hour rolling mean feature was created to capture short-term demand trends. Unlike lagged features, which represent individual historical observations, the rolling mean summarizes average demand behavior over the previous day.

The rolling mean provides contextual information about the recent demand environment and may help forecasting models distinguish between temporary fluctuations and broader consumption trends. Because the feature requires 24 hours of historical observations, the first 23 records contain missing values.

### Cyclical Time Features

Hour-of-day is inherently cyclical, with hour 23 and hour 0 representing adjacent points in time. To preserve this cyclical relationship, sine and cosine transformations were applied to the hour feature.

In [10]:
import numpy as np

features_df['hour_sin'] = np.sin(
    2 * np.pi * features_df['hour'] / 24
)

features_df['hour_cos'] = np.cos(
    2 * np.pi * features_df['hour'] / 24
)

features_df[
    [
        'hour',
        'hour_sin',
        'hour_cos'
    ]
].head(25)

,hour,hour_sin,hour_cos
0,0,0.000000e+00,1.000000e+00
1,1,2.588190e-01,9.659258e-01
2,2,5.000000e-01,8.660254e-01
3,3,7.071068e-01,7.071068e-01
4,4,8.660254e-01,5.000000e-01
5,5,9.659258e-01,2.588190e-01
6,6,1.000000e+00,6.123234e-17
7,7,9.659258e-01,-2.588190e-01
8,8,8.660254e-01,-5.000000e-01
9,9,7.071068e-01,-7.071068e-01


### Cyclical Encoding of Hour

The hour variable was transformed using sine and cosine functions to preserve its cyclical nature. In raw form, hour values range from 0 to 23, causing adjacent hours such as 23:00 and 00:00 to appear numerically distant despite being only one hour apart.

The sine and cosine transformations map hours onto a circular representation, allowing machine learning models to better capture daily periodic patterns. This encoding is widely used in time-series forecasting and helps models learn temporal relationships more effectively.

## Final Modeling Dataset

After completing all feature engineering steps, a final modeling dataset was prepared by removing observations with missing values introduced by lagged and rolling features.

In [11]:
model_df = features_df.dropna(
    subset=[
        'load_lag_1',
        'load_lag_24',
        'load_lag_168',
        'rolling_mean_24'
    ]
).copy()

print("Final model shape:", model_df.shape)

Final model shape: (50232, 29)


### Final Modeling Dataset

Following feature engineering, a final modeling dataset was created by removing observations containing missing values introduced by lagged and rolling features. The resulting dataset contained 50,232 observations and 29 variables.

The final feature set combines original weather, generation, demand, and calendar variables with engineered renewable, lagged, trend-based, and cyclical time features. This dataset provides a comprehensive foundation for electricity demand forecasting and subsequent machine learning model development.

### Verifing Newly Engineered Features

In [12]:
new_features = [
    'renewable_generation',
    'renewable_share_pct',
    'load_lag_1',
    'load_lag_24',
    'load_lag_168',
    'rolling_mean_24',
    'hour_sin',
    'hour_cos'
]

model_df[new_features].describe()

,renewable_generation,renewable_share_pct,load_lag_1,load_lag_24,load_lag_168,rolling_mean_24,hour_sin,hour_cos
count,50232.000000,50232.000000,50232.000000,50232.000000,50232.000000,50232.000000,5.023200e+04,5.023200e+04
mean,16115.597189,28.949695,55506.259496,55507.278090,55496.900721,55507.180130,-1.128081e-17,-5.590899e-17
std,10420.903463,18.206768,10015.672706,10016.561684,10018.434129,6380.478905,7.071138e-01,7.071138e-01
min,312.000000,0.500037,31307.000000,31307.000000,31307.000000,38107.458333,-1.000000e+00,-1.000000e+00
25%,7601.500000,14.325490,47113.750000,47113.750000,47108.000000,50898.260417,-7.071068e-01,-7.071068e-01
50%,14193.500000,25.885440,55115.000000,55115.000000,55094.500000,56600.979167,6.123234e-17,-6.123234e-17
75%,23007.250000,40.258397,64323.250000,64326.000000,64316.250000,60137.927083,7.071068e-01,7.071068e-01
max,61169.000000,98.710884,77549.000000,77549.000000,77549.000000,68365.083333,1.000000e+00,1.000000e+00


### Validation of Engineered Features

Descriptive statistics were used to validate the engineered features prior to model development. All engineered variables contained the expected number of observations, indicating that missing values introduced during feature creation had been successfully addressed.

The renewable energy features exhibited substantial variability, reflecting changing contributions from solar and wind generation. Lagged demand features closely matched the statistical properties of the target variable, confirming their suitability for forecasting applications. The rolling mean feature produced a smoother representation of recent demand trends, while the cyclical hour features displayed the expected bounded range between -1 and 1.

## Exporting the Feature Dataset

The final engineered dataset was exported for use in the forecasting workflow. This dataset will serve as the primary input for machine learning models developed in Notebook 05.

In [13]:
model_df.to_csv(
    "../data/processed/master_features.csv",
    index=False
)

print("master_features.csv saved successfully.")

master_features.csv saved successfully.
